# **EDA — Exploratory Data Analysis and Hypothesis Testing**

## Objectives

* Profile the cleaned dataset and describe the campaign, the client base and the outcome distribution
* Test five stated hypotheses using appropriate statistical methods
* Report effect sizes alongside p-values, since a sample of this size will return significance for trivially small differences
* Test whether call duration is usable as a predictor, or whether it constitutes target leakage
* Save all test results for use in the dashboard

## Inputs

* Data_Set/clean_data/v1/bank_marketing_cleaned.csv — the cleaned dataset produced by Notebook 01

## Outputs

* Data_Set/outputs/v1/hypothesis_results.csv — test statistic, p-value, effect size and outcome for each hypothesis
* Data_Set/outputs/v1/descriptive_summary.csv — summary statistics for the cleaned dataset

## Additional Comments

* With 41,176 records, statistical significance is almost guaranteed for any real
  difference. Every test therefore reports an effect size (Cramér's V for categorical
  associations, rank-biserial correlation for Mann-Whitney tests) so that the practical
  size of each difference can be judged separately from its statistical significance.

* H2 and H5 are framed as null hypotheses: the analysis tests for the absence of
  disparity across education level and the absence of a usable relationship for duration.
  A "not supported" result is a finding about the campaign, not a failure of the analysis.

* Findings describe association only. The data is observational, and no causal claim is
  made about why any group subscribed at a higher or lower rate.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop/fair-marketing-analytics'

# Section 1 — Load and Profile

In [4]:
import pandas as pd
import numpy as np
import os
from scipy import stats

pd.set_option('display.max_columns', None)

version = 'v1'
clean_dir = f'Data_Set/clean_data/{version}'
output_dir = f'Data_Set/outputs/{version}'
os.makedirs(output_dir, exist_ok=True)

print(f"Reading from: {clean_dir}")
print(f"Writing to:   {output_dir}")

Reading from: Data_Set/clean_data/v1
Writing to:   Data_Set/outputs/v1


In [5]:
df = pd.read_csv(f'{clean_dir}/bank_marketing_cleaned.csv')

print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns\n")
print(df.dtypes)

Loaded 41,176 rows and 23 columns

age                    int64
job                   object
marital               object
education             object
housing               object
loan                  object
contact               object
month                 object
day_of_week           object
duration               int64
campaign               int64
pdays                float64
previous               int64
poutcome              object
emp_var_rate         float64
cons_price_idx       float64
cons_conf_idx        float64
euribor3m            float64
nr_employed          float64
default_disclosed      int64
contacted_before       int64
subscribed             int64
age_band              object
dtype: object


## Outcome distribution

The target variable is the proportion of contacted clients who subscribed to a term
deposit. This sets the baseline against which every subgroup difference is judged.

## Numeric variables

In [6]:
target = df['subscribed'].value_counts()
target_pct = df['subscribed'].value_counts(normalize=True)

print(f"Did not subscribe: {target[0]:,} ({target_pct[0]:.2%})")
print(f"Subscribed:        {target[1]:,} ({target_pct[1]:.2%})")
print(f"\nClass ratio: {target[0] / target[1]:.1f} to 1")

Did not subscribe: 36,537 (88.73%)
Subscribed:        4,639 (11.27%)

Class ratio: 7.9 to 1


In [7]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[numeric_cols].describe().T.round(2)
desc['missing'] = df[numeric_cols].isnull().sum()

desc.to_csv(f'{output_dir}/descriptive_summary.csv')
desc

,count,mean,std,min,25%,50%,75%,max,missing
age,41176.0,40.02,10.42,17.00,32.00,38.00,47.00,98.00,0
duration,41176.0,258.32,259.31,0.00,102.00,180.00,319.00,4918.00,0
campaign,41176.0,2.57,2.77,1.00,1.00,2.00,3.00,56.00,0
pdays,1515.0,6.01,3.82,0.00,3.00,6.00,7.00,27.00,39661
previous,41176.0,0.17,0.49,0.00,0.00,0.00,0.00,7.00,0
emp_var_rate,41176.0,0.08,1.57,-3.40,-1.80,1.10,1.40,1.40,0
cons_price_idx,41176.0,93.58,0.58,92.20,93.08,93.75,93.99,94.77,0
cons_conf_idx,41176.0,-40.50,4.63,-50.80,-42.70,-41.80,-36.40,-26.90,0
euribor3m,41176.0,3.62,1.73,0.63,1.34,4.86,4.96,5.04,0
nr_employed,41176.0,5167.03,72.25,4963.60,5099.10,5191.00,5228.10,5228.10,0


## Categorical variables

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"\n{col} — {df[col].nunique()} categories, {df[col].isnull().sum()} missing")
    print(df[col].value_counts(dropna=False).head(12).to_string())


job — 11 categories, 330 missing
job
admin.           10419
blue-collar       9253
technician        6739
services          3967
management        2924
retired           1718
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
NaN                330

marital — 3 categories, 80 missing
marital
married     24921
single      11564
divorced     4611
NaN            80

education — 7 categories, 1730 missing
education
university.degree      12164
high.school             9512
basic.9y                6045
professional.course     5240
basic.4y                4176
basic.6y                2291
NaN                     1730
illiterate                18

housing — 2 categories, 990 missing
housing
yes    21571
no     18615
NaN      990

loan — 2 categories, 990 missing
loan
no     33938
yes     6248
NaN      990

contact — 2 categories, 0 missing
contact
cellular     26135
telephone    15041

month — 10 categories, 0 missing
month
may  

## Subscription rate by group

The overall subscription rate is 11.3%. This section reports the rate within each
category of the demographic and financial fields, which is the basis for the hypothesis
tests in Section 2.

In [9]:
def rate_by(col):
    out = df.groupby(col, observed=True, dropna=False).agg(
        contacted=('subscribed', 'size'),
        subscribed=('subscribed', 'sum')
    )
    out['rate'] = (out['subscribed'] / out['contacted'] * 100).round(2)
    return out.sort_values('rate', ascending=False)

for col in ['age_band', 'job', 'education', 'marital', 'housing', 'loan']:
    print(f"\n=== {col} ===")
    print(rate_by(col).to_string())


=== age_band ===
             contacted  subscribed   rate
age_band                                 
60 and over       1192         472  39.60
Under 25          1067         256  23.99
25-59            38917        3911  10.05

=== job ===
               contacted  subscribed   rate
job                                        
student              875         275  31.43
retired             1718         434  25.26
unemployed          1014         144  14.20
admin.             10419        1351  12.97
management          2924         328  11.22
NaN                  330          37  11.21
technician          6739         730  10.83
self-employed       1421         149  10.49
housemaid           1060         106  10.00
entrepreneur        1456         124   8.52
services            3967         323   8.14
blue-collar         9253         638   6.90

=== education ===
                     contacted  subscribed   rate
education                                        
illiterate              

### Findings

- The overall subscription rate is 11.27%, against a class ratio of roughly 8 to 1.
- Age shows the largest variation of any field. Clients aged 60 and over subscribed at
  39.60%, and those under 25 at 23.99%, against 10.05% for the 25 to 59 group. The two
  age groups most likely to be treated as potentially vulnerable under UK conduct rules
  are the two most responsive to the campaign.
- Job category reflects the same pattern, with students at 31.43% and retired clients at
  25.26%. These groups overlap substantially with the outer age bands and should not be
  read as independent findings.
- Education shows a consistent gradient, from 13.72% for university degree to 7.82% for
  basic 9y schooling. The "illiterate" category contains only 18 records and is too small
  to interpret.
- Existing housing and personal loans show almost no variation in subscription rate
  (11.62% against 10.88%, and 11.34% against 10.93%). Any statistical significance found
  in Section 2 for these fields will reflect sample size rather than practical importance.
- Records with missing values do not subscribe at unusual rates, so the missingness does
  not appear to bias the outcome.

---

# Section 2 — Hypothesis Testing

## Method

Five hypotheses are tested. Categorical associations use the chi-square test of
independence, with Cramér's V as the effect size. Comparisons of a numeric variable
across the two outcome groups use the Mann-Whitney U test, chosen over the t-test because
the campaign variables are heavily skewed and not normally distributed, with rank-biserial
correlation as the effect size.

Significance is assessed at alpha = 0.05.

### Why effect sizes are reported throughout

With 41,176 records, the chi-square test will return a significant result for differences
far too small to act on. Reporting p-values alone would therefore overstate every finding.
Each test reports an effect size so that the practical size of a difference can be judged
separately from whether it is statistically detectable. Cramér's V is interpreted here as
negligible below 0.10, small to 0.20, moderate to 0.30, and large above that.

### Scope and limitations

The data is observational. Clients were not randomly assigned to groups, and the tests
identify association only. No causal claim is made about why any group subscribed at a
higher or lower rate, and differences may reflect who the bank chose to call as much as
how clients responded.

### Fields deliberately not tested

Job category is not tested as a separate hypothesis. Students and retired clients overlap
substantially with the under-25 and 60-and-over age bands, so a significant result would
largely restate the age finding rather than add to it. Reporting both as independent
findings would overstate the evidence.

The "illiterate" education category contains 18 records. It is retained in the data but is
not interpreted, as any rate calculated from a cell that size is unstable.

In [11]:
def cramers_v(contingency):
    """Cramér's V effect size for a chi-square test of independence."""
    chi2 = stats.chi2_contingency(contingency)[0]
    n = contingency.values.sum()
    min_dim = min(contingency.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))


def interpret_v(v):
    """Plain-language interpretation of a Cramér's V value."""
    if v < 0.10:
        return 'negligible'
    elif v < 0.20:
        return 'small'
    elif v < 0.30:
        return 'moderate'
    return 'large'


def rank_biserial(group_a, group_b):
    """Rank-biserial correlation effect size for a Mann-Whitney U test."""
    u = stats.mannwhitneyu(group_a, group_b, alternative='two-sided')[0]
    return 1 - (2 * u) / (len(group_a) * len(group_b))


results = []

## H1 — Subscription rate differs by age band

Null: subscription rate is independent of age band.
Alternative: subscription rate differs across age bands.

Age is a protected characteristic under the Equality Act 2010. This test establishes
whether the campaign outcome varied by age, which determines whether a model trained on
this data would inherit an age-related pattern.

In [12]:
ct = pd.crosstab(df['age_band'], df['subscribed'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
v = cramers_v(ct)

print(ct, "\n")
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.2e}")
print(f"Cramér's V = {v:.3f} ({interpret_v(v)})")

results.append({
    'hypothesis': 'H1',
    'statement': 'Subscription rate differs by age band',
    'test': 'Chi-square test of independence',
    'statistic': round(chi2, 2),
    'p_value': p,
    'effect_size': round(v, 3),
    'effect_measure': "Cramér's V",
    'interpretation': interpret_v(v),
    'outcome': 'Supported' if p < 0.05 else 'Not supported'
})

subscribed       0     1
age_band                
25-59        35006  3911
60 and over    720   472
Under 25       811   256 

chi2 = 1187.53, dof = 2, p = 1.35e-258
Cramér's V = 0.170 (small)


## H2 — Subscription rate does not differ by education level

Null: subscription rate is independent of education level.
Alternative: subscription rate differs across education levels.

This hypothesis is framed as a null. Education is not itself a protected characteristic,
but it acts as a proxy for socioeconomic status. The test asks whether the campaign
produced different outcomes across that proxy. A result of "not supported" would be a
finding about the campaign, not a failure of the analysis.

In [13]:
ct = pd.crosstab(df['education'], df['subscribed'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
v = cramers_v(ct)

print(ct, "\n")
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.2e}")
print(f"Cramér's V = {v:.3f} ({interpret_v(v)})")

results.append({
    'hypothesis': 'H2',
    'statement': 'Subscription rate does not differ by education level',
    'test': 'Chi-square test of independence',
    'statistic': round(chi2, 2),
    'p_value': p,
    'effect_size': round(v, 3),
    'effect_measure': "Cramér's V",
    'interpretation': interpret_v(v),
    'outcome': 'Not supported' if p < 0.05 else 'Supported'
})

subscribed               0     1
education                       
basic.4y              3748   428
basic.6y              2103   188
basic.9y              5572   473
high.school           8481  1031
illiterate              14     4
professional.course   4645   595
university.degree    10495  1669 

chi2 = 175.80, dof = 6, p = 2.65e-35
Cramér's V = 0.067 (negligible)


## H3 — Clients with an existing housing loan are less likely to subscribe

Null: subscription rate is independent of housing loan status.
Alternative: subscription rate differs by housing loan status.

Section 1 showed rates of 11.62% and 10.88% for these groups. This test is retained
specifically to demonstrate the distinction between statistical significance and practical
importance at this sample size.

In [14]:
sub = df[df['housing'].notna()]
ct = pd.crosstab(sub['housing'], sub['subscribed'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
v = cramers_v(ct)

print(ct, "\n")
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}")
print(f"Cramér's V = {v:.3f} ({interpret_v(v)})")

results.append({
    'hypothesis': 'H3',
    'statement': 'Subscription rate differs by housing loan status',
    'test': 'Chi-square test of independence',
    'statistic': round(chi2, 2),
    'p_value': p,
    'effect_size': round(v, 3),
    'effect_measure': "Cramér's V",
    'interpretation': interpret_v(v),
    'outcome': 'Supported' if p < 0.05 else 'Not supported'
})

subscribed      0     1
housing                
no          16590  2025
yes         19064  2507 

chi2 = 5.45, dof = 1, p = 0.0196
Cramér's V = 0.012 (negligible)


## H4 — Repeated contact within a campaign is associated with lower subscription

Null: the number of contacts does not differ between subscribers and non-subscribers.
Alternative: the number of contacts differs between the two groups.

This is a conduct hypothesis as much as a commercial one. Repeatedly calling clients who
continue to decline raises a fair treatment question independently of whether it is
commercially effective.

In [15]:
sub_yes = df[df['subscribed'] == 1]['campaign']
sub_no = df[df['subscribed'] == 0]['campaign']

u, p = stats.mannwhitneyu(sub_yes, sub_no, alternative='two-sided')
r = rank_biserial(sub_yes, sub_no)

print(f"Median contacts, subscribed:     {sub_yes.median():.0f}")
print(f"Median contacts, not subscribed: {sub_no.median():.0f}")
print(f"Max contacts to a single client: {df['campaign'].max()}")
print(f"Clients contacted 10+ times:     {(df['campaign'] >= 10).sum():,}\n")
print(f"U = {u:.0f}, p = {p:.2e}")
print(f"Rank-biserial r = {r:.3f}")

results.append({
    'hypothesis': 'H4',
    'statement': 'Number of contacts differs between subscribers and non-subscribers',
    'test': 'Mann-Whitney U',
    'statistic': round(u, 0),
    'p_value': p,
    'effect_size': round(r, 3),
    'effect_measure': 'Rank-biserial r',
    'interpretation': 'see note',
    'outcome': 'Supported' if p < 0.05 else 'Not supported'
})

Median contacts, subscribed:     2
Median contacts, not subscribed: 2
Max contacts to a single client: 56
Clients contacted 10+ times:     1,094

U = 75391970, p = 3.63e-38
Rank-biserial r = 0.110


## H5 — Call duration cannot serve as a legitimate predictor

Null: call duration does not differ between subscribers and non-subscribers.
Alternative: call duration differs between the two groups.

The dataset documentation states that duration is not known before a call is made, that
the outcome is known once the call ends, and that the field should be discarded for a
realistic predictive model.

This hypothesis tests that statement rather than accepting it. If duration is strongly
associated with the outcome, it constitutes target leakage: the field encodes the answer
rather than predicting it. A model containing it would appear highly accurate in testing
and would be unusable in practice, because the value is unavailable at the moment the
targeting decision is taken.

This is a model governance question. A leaked feature is the most common way an unusable
model passes internal validation.

In [16]:
dur_yes = df[df['subscribed'] == 1]['duration']
dur_no = df[df['subscribed'] == 0]['duration']

u, p = stats.mannwhitneyu(dur_yes, dur_no, alternative='two-sided')
r = rank_biserial(dur_yes, dur_no)
pb = stats.pointbiserialr(df['subscribed'], df['duration'])

print(f"Median duration, subscribed:     {dur_yes.median():.0f} seconds")
print(f"Median duration, not subscribed: {dur_no.median():.0f} seconds")
print(f"Calls of 0 seconds: {(df['duration'] == 0).sum()}\n")
print(f"U = {u:.0f}, p = {p:.2e}")
print(f"Rank-biserial r = {r:.3f}")
print(f"Point-biserial correlation = {pb.correlation:.3f}")

results.append({
    'hypothesis': 'H5',
    'statement': 'Call duration cannot serve as a legitimate predictor (leakage)',
    'test': 'Mann-Whitney U and point-biserial correlation',
    'statistic': round(u, 0),
    'p_value': p,
    'effect_size': round(r, 3),
    'effect_measure': 'Rank-biserial r',
    'interpretation': 'leakage confirmed' if abs(r) > 0.3 else 'leakage not confirmed',
    'outcome': 'Supported' if p < 0.05 and abs(r) > 0.3 else 'Not supported'
})

Median duration, subscribed:     449 seconds
Median duration, not subscribed: 164 seconds
Calls of 0 seconds: 4

U = 138721931, p = 0.00e+00
Rank-biserial r = -0.637
Point-biserial correlation = 0.405


In [17]:
results_df = pd.DataFrame(results)
results_df['p_value'] = results_df['p_value'].apply(
    lambda x: f"{x:.2e}" if x < 0.001 else f"{x:.4f}"
)

results_df.to_csv(f'{output_dir}/hypothesis_results.csv', index=False)

print(f"Saved to {output_dir}/hypothesis_results.csv\n")
results_df[['hypothesis', 'statement', 'p_value', 'effect_size',
            'interpretation', 'outcome']]

Saved to Data_Set/outputs/v1/hypothesis_results.csv



,hypothesis,statement,p_value,effect_size,interpretation,outcome
0,H1,Subscription rate differs by age band,1.35e-258,0.170,small,Supported
1,H2,Subscription rate does not differ by education...,2.65e-35,0.067,negligible,Not supported
2,H3,Subscription rate differs by housing loan status,0.0196,0.012,negligible,Supported
3,H4,Number of contacts differs between subscribers...,3.63e-38,0.110,see note,Supported
4,H5,Call duration cannot serve as a legitimate pre...,0.00e+00,-0.637,leakage confirmed,Supported


## Summary of results

### Findings

**H1 — Supported.** Subscription rate differs by age band (chi-square, p = 1.35e-258,
Cramér's V = 0.170, small). Clients aged 60 and over subscribed at 39.60% and those under
25 at 23.99%, against 10.05% for the 25 to 59 group. The effect size is small in absolute
terms, but the disparity is the largest in the dataset and is the pattern a model trained
on this data would learn most strongly.

**H2 — Not supported.** The hypothesis that subscription rate does not differ by education
level is rejected (p = 2.65e-35, V = 0.067, negligible). A gradient is present, from 13.72%
for university degree to 7.82% for basic 9y schooling, but the effect size indicates
education explains very little of the variation on its own.

**H3 — Supported statistically, not practically.** Housing loan status is significantly
associated with subscription (p = 0.0196), but Cramér's V of 0.012 is negligible: the rates
are 11.62% and 10.88%, a difference of 0.74 percentage points. This result is retained
because it demonstrates the central methodological point of this analysis. At 41,176
records, statistical significance can be achieved by differences too small to act on, and
reporting the p-value alone would materially misrepresent the finding.

**H4 — Supported.** The number of contacts differs between subscribers and non-subscribers
(Mann-Whitney U, p = 3.63e-38, rank-biserial r = 0.110). Repeated contact is associated with
lower subscription, which raises a fair treatment question separate from commercial
effectiveness.

**H5 — Supported. Leakage confirmed.** Call duration differs substantially between the two
groups (p < 0.001, rank-biserial r = −0.637, point-biserial correlation 0.405). Median
duration was 449 seconds for subscribers against 164 seconds for non-subscribers, and all
four zero-second calls resulted in no subscription. Duration is not available at the point
a targeting decision is made and is only known once the outcome is effectively determined.
It is therefore excluded from all models in Notebook 04, and a comparison model retaining
it is built solely to quantify the distortion it introduces.

### What these results mean for the modelling stage

Two findings shape Notebook 04 directly.

First, the age pattern is the strongest signal in the data. Any model that maximises
subscription probability will preferentially target clients aged 60 and over and under 25.
Both groups are more likely to be treated as potentially vulnerable under FCA rules. A model
can therefore be accurate, commercially effective, and still produce an outcome a conduct
function would not approve. This is tested directly through subgroup fairness metrics in
Notebook 04.

Second, H3 establishes that significance and importance are separate questions. The same
distinction applies to model performance: a model achieving 88.7% accuracy on this data is
performing no better than always predicting "no", given the class imbalance. Accuracy is
therefore not used as a headline metric.

### Limitations

- The findings are associational. Clients were not randomly assigned to groups, and
  differences may reflect who the bank chose to contact as much as how clients responded.
- Age band and job category overlap substantially. They are not independent findings.
- The campaign ran from May 2008 to November 2010, spanning the financial crisis. Interest
  rate and employment conditions during that period were unusual, which limits how far the
  patterns transfer to a present-day UK campaign.
- The data originates from a Portuguese bank. It is treated here as a proxy for a UK
  retail campaign, and the legal assessment is conducted on that basis rather than as a
  description of an actual UK firm.

---

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
